# Stress prediction: preprocessing and training all six models
## Viva study notebook
Read the explanations and executed outputs from top to bottom. The default run **loads and evaluates the website's existing checkpoints**. The training cells contain real, runnable fitting code; set `RUN_RETRAIN=True` only to create a separate fresh training run. Existing models and CSVs are never overwritten or activated by this notebook.

**Data origin:** the current 10,000 profiles and stress labels are an educational generated-data simulation, using 19 survey-style inputs and label-noise standard deviation 0.25. These are not independently observed respondents or a verified Kaggle dataset. Accuracy here measures recovery of a hypothetical rule; it does not establish real-world or clinical stress-prediction accuracy. Reducing simulation label noise changed the learning problem; it was not an improvement on unchanged real BRFSS outcomes.

**Workflow:** CSV → verify IDs, targets and stored splits → fit preprocessing on 6,300 fitting rows → fit six model families → use 700 development rows for neural checkpoint selection → validation review → freeze checkpoints → evaluate on the reused 1,500-row internal holdout → website API. The 1,500 validation rows are separate from development. No genuine temporal measurements are present.

In [1]:
from pathlib import Path
import sys, json, hashlib, copy, time, inspect, platform
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'models/training_preview.json').exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import joblib, torch, sklearn, xgboost
from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, log_loss
from src.training_preview_inference import TrainingPreviewPredictor, NAMES, SEEDS
from src.training_variants import FeatureEncoder, PositionConv1D
from src.models import LSTMNet, CNNLSTMNet
torch.set_num_threads(4)
RUN_RETRAIN = False
engine = TrainingPreviewPredictor(ROOT)
BASE = engine.root
def digest(path):
    with Path(path).open('rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()
DATA = BASE / engine.manifest['csv']
assert digest(DATA) == engine.manifest['sha256']
print('Python:', platform.python_version())
print('Versions:', dict(numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, torch=torch.__version__, xgboost=xgboost.__version__))
print('CSV:', DATA.relative_to(ROOT))
print('SHA256:', digest(DATA))
print('Checkpoint revision:', engine.model_revision)
print('Mode:', 'Fresh isolated retraining' if RUN_RETRAIN else 'Evaluation of saved active checkpoints; training cells guarded')

Python: 3.12.15
Versions: {'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.6.1', 'torch': '2.14.1+cpu', 'xgboost': '3.4.1'}
CSV: model_training/base_models/model_training_data.csv
SHA256: e26075ea7b48e3413bced84d97a8d85c8e7aff76e81d0803f1a560e0448b7837
Checkpoint revision: rf_cnn_tuned_20261026
Mode: Evaluation of saved active checkpoints; training cells guarded


## 1. Read and audit the CSV
The target is ordered: 0 = rarely/never, 1 = sometimes, 2 = usually/always. `sample_id`, `data_origin`, `split` and `target` must never enter the feature matrix. Identical feature values can occur for different people in survey data; an ID audit and a feature-duplicate audit answer different questions. Here these are generated profile IDs, not respondent identities.

In [2]:
df = pd.read_csv(DATA)
FEATURES = engine.schema['numeric'] + engine.schema['categorical']
assert len(FEATURES) == 19
assert df.sample_id.is_unique
assert not df[FEATURES].isna().any().any()
assert set(df.target.unique()) == {0, 1, 2}
print('Rows:', len(df), '| input features:', len(FEATURES))
print('Origin:', df.data_origin.unique().tolist())
print('Numeric inputs:', engine.schema['numeric'])
print('Categorical inputs:', engine.schema['categorical'])
print('Missing inputs:', int(df[FEATURES].isna().sum().sum()))
print('Duplicate complete input profiles:', int(df.duplicated(FEATURES).sum()))
print(pd.crosstab(df.split, df.target).to_string())
print('First two feature rows:')
print(df[FEATURES].head(2).to_string(index=False))

Rows: 10000 | input features: 19
Origin: ['synthetic_simulation']
Numeric inputs: ['age', 'sleep_duration', 'unhealthy_days', 'bmi']
Categorical inputs: ['sex', 'exercise', 'general_health', 'marital', 'education', 'employment', 'income', 'support', 'isolation', 'job_loss', 'food_insecurity', 'bill_difficulty', 'utility_threat', 'transport_barrier', 'medical_cost']
Missing inputs: 0
Duplicate complete input profiles: 0
target         0     1     2
split                       
test         521   590   389
train       2475  2704  1821
validation   512   602   386
First two feature rows:
 age  sleep_duration  unhealthy_days   bmi  sex  exercise  general_health  marital  education  employment  income  support  isolation  job_loss  food_insecurity  bill_difficulty  utility_threat  transport_barrier  medical_cost
  59               8               0 25.52    1         1               3        6          4           8       6        3          2         2                3                2    

## 2. Preserve the existing splits
Do not create new favorable splits. Saved training roles reproduce the exact fitting/development row order. Preprocessing sees fitting rows only. Development selects neural epochs; validation reviews the chosen candidate. The test partition has already informed earlier experiments, so it is a **reused internal holdout**, not an untouched final test.

In [3]:
roles = pd.read_csv(BASE / 'training_roles.csv')
indexed = df.set_index('sample_id', drop=False)
fit = indexed.loc[roles.loc[roles.role.eq('fit'), 'sample_id']].reset_index(drop=True)
dev = indexed.loc[roles.loc[roles.role.eq('early_stop'), 'sample_id']].reset_index(drop=True)
validation = df[df.split.eq('validation')].copy()
test = df[df.split.eq('test')].copy()
groups = [set(f.sample_id) for f in (fit, dev, validation, test)]
assert all(not groups[i] & groups[j] for i in range(4) for j in range(i))
assert groups[0] | groups[1] == set(df[df.split.eq('train')].sample_id)
assert sum(map(len, groups)) == len(df)
print('Fit:', len(fit), '| development:', len(dev), '| validation:', len(validation), '| internal holdout:', len(test))
print('All four ID groups are disjoint.')

Fit: 6300 | development: 700 | validation: 1500 | internal holdout: 1500
All four ID groups are disjoint.


## 3. Preprocessing, without leakage
For LR, XGBoost and the three neural families: standardize four numeric inputs using fitting-set means/standard deviations, and one-hot encode the 15 categorical inputs with fixed schema categories. This produces **69** columns. Categories come from the input schema, not the target.

The improved Random Forest uses **33** columns: 19 coded original inputs plus 14 explicitly defined contextual aggregates (sleep deficits, unhealthy-day transforms and social/material-risk combinations). The following is the exact encoder implementation. These aggregates are hypothetical, not validated health scales. No imputation is required for this complete CSV; unknown/missing production values are rejected rather than silently treated as known categories.

In [4]:
print(inspect.getsource(FeatureEncoder))
standard_encoder = FeatureEncoder(engine.schema, 'original').fit(fit[FEATURES])
forest_encoder = FeatureEncoder(engine.schema, 'engineered').fit(fit[FEATURES])
print('Standard fitting shape:', standard_encoder.transform(fit[FEATURES]).shape)
print('Forest fitting shape:', forest_encoder.transform(fit[FEATURES]).shape)
print('Training-only numeric means:', standard_encoder.base.named_transformers_['numeric'].mean_)
for name, bundles in engine.models.items():
    for pre, model in bundles:
        current = pre.transform(test[FEATURES]).astype('float32')
        expected = forest_encoder.transform(test[FEATURES]) if name == 'Random Forest' else standard_encoder.transform(test[FEATURES])
        np.testing.assert_allclose(current, expected, rtol=1e-6, atol=1e-6)
print('Fresh fit-only preprocessing matches every active saved encoder.')

class FeatureEncoder:
    def __init__(self,schema,mode='original'):self.schema=schema;self.mode=mode
    def fit(self,frame):
        s=self.schema;cats=[list(range(s['limits'][k][0],s['limits'][k][1]+1)) for k in s['categorical']]
        self.base=ColumnTransformer([('numeric',StandardScaler(),s['numeric']),('categorical',OneHotEncoder(categories=cats,sparse_output=False,handle_unknown='error'),s['categorical'])]).fit(frame)
        self.names=s['numeric']+s['categorical'];return self
    def transform(self,f):
        if self.mode=='original':return self.base.transform(f).astype('float32')
        if self.mode!='engineered':raise ValueError('Unknown encoder mode.')
        # Hypothetical equal-weight contextual aggregates; not a validated scale
        # and not the simulator's exact label equation. No target/ID is used.
        support=(f.support-1)/4;isolation=(5-f.isolation)/4;food=(5-f.food_insecurity)/4;health=(f.general_health-1)/4
        material=sum((f[k]==1).astype(float)

## 4. Inspect actual fitted hyperparameters and architectures
Hyperparameters control model capacity and optimization; learned parameters are fitted from data. LR learns class coefficients; Random Forest averages trees; XGBoost adds boosted trees. CNN learns feature-position filters; LSTM uses recurrent processing over a fixed tabular feature order; CNN-LSTM combines those operations. **The LSTM does not learn personal temporal history**, because the CSV has no longitudinal sequences.

Each neural family averages probabilities from two fixed seeds (20261026 and 20261027); it is not the best of two runs chosen after viewing the test. The CNN now preserves feature positions and adds a linear skip connection. Other architectures are imported directly from the same source used by inference.

In [5]:
for name in NAMES[:3]:
    print('\n' + name)
    print(json.dumps(engine.models[name][0][1].get_params(), indent=2, default=str))
for name in NAMES[3:]:
    print('\n' + name + ' architecture:')
    print(engine.models[name][0][1])
print('Neural seeds:', SEEDS)


Logistic Regression
{
  "C": 0.1,
  "class_weight": {
    "0": 0.9708498192994385,
    "1": 0.9309641135766961,
    "2": 1.1284941693946817
  },
  "dual": false,
  "fit_intercept": true,
  "intercept_scaling": 1,
  "l1_ratio": null,
  "max_iter": 1200,
  "multi_class": "deprecated",
  "n_jobs": 4,
  "penalty": "l2",
  "random_state": 20261026,
  "solver": "lbfgs",
  "tol": 0.0001,
  "verbose": 0,
  "warm_start": false
}

Random Forest
{
  "bootstrap": true,
  "ccp_alpha": 0.0,
  "class_weight": "balanced",
  "criterion": "gini",
  "max_depth": null,
  "max_features": 0.85,
  "max_leaf_nodes": null,
  "max_samples": 0.9,
  "min_impurity_decrease": 0.0,
  "min_samples_leaf": 2,
  "min_samples_split": 2,
  "min_weight_fraction_leaf": 0.0,
  "monotonic_cst": null,
  "n_estimators": 450,
  "n_jobs": 4,
  "oob_score": false,
  "random_state": 20261026,
  "verbose": 0,
  "warm_start": false
}

XGBoost
{
  "objective": "multi:softprob",
  "base_score": null,
  "booster": null,
  "callbacks": 

## 5. Training setup and protected outputs
The following code trains fresh estimators using the active configurations. Classical models use one seed. Neural networks use two seeds and a weighted cross-entropy loss. Weights are the square root of inverse training class prevalence; they are computed from the fitting labels only. LR stores these weights in its configuration; XGBoost receives sample weights; the improved forest uses its saved balanced-class setting.

AdamW updates neural weights in batches of 256. Gradient clipping limits unstable updates. Unweighted development log-loss chooses the best epoch; patience stops stalled training. CNN uses a learning-rate scheduler. The time limits reproduce the bounded local training approach, so exact retraining results can vary with machine speed and library versions. Fresh outputs remain separate, with no website activation.

In [6]:
y_fit = fit.target.to_numpy(copy=True)
y_dev = dev.target.to_numpy(copy=True)
class_weights = (len(y_fit) / (3 * np.bincount(y_fit, minlength=3))) ** 0.5
print('Training class weights:', class_weights)
FRESH = {}
OUTPUT = None
if RUN_RETRAIN:
    from datetime import datetime
    OUTPUT = ROOT / 'model_training' / ('study_run_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    OUTPUT.mkdir(exist_ok=False)
    (OUTPUT / 'protocol.json').write_text(json.dumps(dict(data_sha256=digest(DATA), seeds=SEEDS,
        fit_ids=fit.sample_id.tolist(), dev_ids=dev.sample_id.tolist(),
        selection='development log-loss only; no test tuning', origin='generated simulation'), indent=2))
    print('Fresh run:', OUTPUT)

def probabilities(model, x):
    if not isinstance(model, torch.nn.Module):
        return np.asarray(model.predict_proba(x), dtype=np.float64)
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(torch.from_numpy(x[a:a+256])).softmax(1).numpy()
                               for a in range(0, len(x), 256)])

def train_family(name):
    if not RUN_RETRAIN:
        print(name + ': training code ready; fresh fitting disabled. Saved checkpoint evaluation follows.')
        return
    bundles = []
    for seed in (SEEDS if name in NAMES[3:] else SEEDS[:1]):
        np.random.seed(seed); torch.manual_seed(seed)
        pre = FeatureEncoder(engine.schema, 'engineered' if name == 'Random Forest' else 'original').fit(fit[FEATURES])
        x = pre.transform(fit[FEATURES]).astype('float32')
        dx = pre.transform(dev[FEATURES]).astype('float32')
        start = time.monotonic(); history = []; best_epoch = None
        if name in NAMES[:3]:
            model = clone(engine.models[name][0][1])
            model.set_params(random_state=seed)
            if 'n_jobs' in model.get_params(): model.set_params(n_jobs=4)
            if name == 'Logistic Regression': model.set_params(class_weight=dict(enumerate(class_weights)))
            model.fit(x, y_fit, **({'sample_weight': class_weights[y_fit]} if name == 'XGBoost' else {}))
        else:
            cnn = name == '1D CNN'
            model = PositionConv1D(x.shape[1], channels=16, hidden=64, dropout=.15) if cnn else (LSTMNet(x.shape[1]) if name == 'LSTM' else CNNLSTMNet(x.shape[1]))
            optimizer = torch.optim.AdamW(model.parameters(), lr=.001 if cnn else .0007, weight_decay=.0003 if cnn else .001)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=.5, patience=3) if cnn else None
            criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32))
            loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.from_numpy(x), torch.from_numpy(y_fit)), batch_size=256, shuffle=True)
            best_loss = float('inf'); state = None; stale = 0
            for epoch in range(1, (80 if cnn else 40) + 1):
                model.train(); total_loss = 0.
                for bx, by in loader:
                    optimizer.zero_grad()
                    loss = criterion(model(bx), by)
                    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                    optimizer.step(); total_loss += float(loss.detach()) * len(by)
                dev_loss = float(log_loss(y_dev, probabilities(model, dx), labels=[0,1,2]))
                history.append(dict(epoch=epoch, training_loss=total_loss/len(y_fit), development_logloss=dev_loss))
                if dev_loss < best_loss - 1e-5:
                    best_loss = dev_loss; state = copy.deepcopy(model.state_dict()); best_epoch = epoch; stale = 0
                else: stale += 1
                if scheduler: scheduler.step(dev_loss)
                if stale >= (10 if cnn else 6) or time.monotonic()-start >= (180 if cnn else 120): break
            model.load_state_dict(state); model.eval()
        stem = OUTPUT / (name.replace(' ', '_') + '_' + str(seed))
        joblib.dump(pre, str(stem) + '.encoder.joblib')
        if name in NAMES[:3]: joblib.dump(model, str(stem) + '.joblib')
        else: torch.save(model.state_dict(), str(stem) + '.pt')
        (Path(str(stem)+'.history.json')).write_text(json.dumps(dict(seed=seed, best_epoch=best_epoch, history=history, seconds=time.monotonic()-start), indent=2))
        bundles.append((pre, model))
        print(name, '| seed', seed, '| best epoch', best_epoch, '| seconds', round(time.monotonic()-start, 1))
    FRESH[name] = bundles

Training class weights: [0.97084982 0.93096411 1.12849417]


## Train: Logistic Regression
Fit a linear multiclass probability model on standardized/one-hot inputs. Regularization limits coefficient size.

In [7]:
train_family('Logistic Regression')

Logistic Regression: training code ready; fresh fitting disabled. Saved checkpoint evaluation follows.


## Train: Random Forest
Fit 450 trees on the 33-column engineered representation. Depth is unrestricted, minimum leaf size is 2, and balanced class weights protect minority-class learning.

In [8]:
train_family('Random Forest')

Random Forest: training code ready; fresh fitting disabled. Saved checkpoint evaluation follows.


## Train: XGBoost
Fit the saved boosted-tree configuration with training-only sample weights. This cell reproduces the active model rather than launching a new hyperparameter search.

In [9]:
train_family('XGBoost')

XGBoost: training code ready; fresh fitting disabled. Saved checkpoint evaluation follows.


## Train: 1D CNN
Fit position-preserving convolution layers, a dense head and a linear skip path. Two seeds; learning-rate scheduling; checkpoint chosen on development loss.

In [10]:
train_family('1D CNN')

1D CNN: training code ready; fresh fitting disabled. Saved checkpoint evaluation follows.


## Train: LSTM
Fit the saved bidirectional LSTM with attention pooling and a direct input skip. The ordered inputs are tabular positions, not time points.

In [11]:
train_family('LSTM')

LSTM: training code ready; fresh fitting disabled. Saved checkpoint evaluation follows.


## Train: CNN-LSTM
Convolve and pool the fixed feature vector, then process the resulting positions with an LSTM and a classification head.

In [12]:
train_family('CNN-LSTM')

CNN-LSTM: training code ready; fresh fitting disabled. Saved checkpoint evaluation follows.


## 6. Original training evidence
The recorded histories below come from the actual training runs that produced the currently loaded artifacts. They are not freshly fitted models from this notebook's default execution. The CNN/forest replacement records live in `improved_models/`; the other families use `base_models/`. A best epoch is selected using development loss, not test accuracy.

In [13]:
saved_base = json.loads((BASE / 'validation_metrics.json').read_text())
saved_improved = json.loads((ROOT / 'model_training/improved_models/validation.json').read_text())
for name in NAMES:
    infos = saved_improved[name]['fit'] if name in saved_improved else [item['fit'] for item in saved_base[name]['per_seed']]
    for item in infos:
        history = item.get('history', [])
        print(name, '| seed', item.get('seed'), '| best epoch', item.get('best_epoch'),
              '| recorded epochs', len(history), '| fit seconds', round(item['seconds'], 2))
        if history: print('  First:', history[0], '\n  Last:', history[-1])

Logistic Regression | seed 20261026 | best epoch None | recorded epochs 0 | fit seconds 4.27
Random Forest | seed 20261026 | best epoch None | recorded epochs 0 | fit seconds 5.38
XGBoost | seed 20261026 | best epoch None | recorded epochs 0 | fit seconds 0.75
1D CNN | seed 20261026 | best epoch 9 | recorded epochs 19 | fit seconds 2.45
  First: {'epoch': 1, 'train_weighted_loss': 0.5938221215255677, 'dev_logloss': 0.4101602513625793} 
  Last: {'epoch': 19, 'train_weighted_loss': 0.1616966678225805, 'dev_logloss': 0.21764557240724697}
1D CNN | seed 20261027 | best epoch 9 | recorded epochs 19 | fit seconds 2.67
  First: {'epoch': 1, 'train_weighted_loss': 0.6734066073288993, 'dev_logloss': 0.5364613260065624} 
  Last: {'epoch': 19, 'train_weighted_loss': 0.18588486642118485, 'dev_logloss': 0.21483727779151643}
LSTM | seed 20261026 | best epoch 16 | recorded epochs 22 | fit seconds 19.02
  First: {'epoch': 1, 'development_logloss': 1.0713155512001202} 
  Last: {'epoch': 22, 'development

## 7. Evaluate all six current models: real outputs from checkpoint inference
Each individual label is the largest raw probability (`argmax`). Each neural family's two seed probabilities are averaged first. We report accuracy together with sometimes and high-stress recall so a majority-class shortcut is visible. The reference baseline is the most common **training** class, evaluated on the same holdout. These are generated-profile scores on a reused internal holdout.

The fresh-run branch evaluates all six only after their training has finished. It does not replace production, choose alternative candidates from test results, or overwrite the default saved results.

In [14]:
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score
models = FRESH if RUN_RETRAIN else engine.models
assert set(models) == set(NAMES)
yy = test.target.to_numpy(); predictions = {}; rows = []
for name in NAMES:
    pp = [probabilities(model, pre.transform(test[FEATURES]).astype('float32')) for pre, model in models[name]]
    p = np.mean(pp, axis=0, dtype=np.float64)
    p = p / p.sum(axis=1, keepdims=True)
    pred = p.argmax(axis=1); predictions[name] = p
    recalls = recall_score(yy, pred, labels=[0,1,2], average=None, zero_division=0)
    result = dict(Model=name, Accuracy=accuracy_score(yy,pred), Macro_F1=f1_score(yy,pred,average='macro'),
                  Sometimes_recall=recalls[1], High_recall=recalls[2], QWK=cohen_kappa_score(yy,pred,weights='quadratic'),
                  Macro_AUC=roc_auc_score(yy,p,multi_class='ovr'), High_detected=int(((yy==2)&(pred==2)).sum()), High_missed=int(((yy==2)&(pred!=2)).sum()))
    if not RUN_RETRAIN:
        assert abs(result['Accuracy'] - engine.results['models'][name]['raw']['accuracy']) < 1e-9
    rows.append(result)
results = pd.DataFrame(rows)
print(results.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
majority = int(fit.target.mode().iloc[0]); baseline = np.full(len(yy), majority)
print('\nTraining-majority baseline class:', majority, '| holdout accuracy:', f'{accuracy_score(yy,baseline):.2%}')
print('Baseline sometimes/high recall:', recall_score(yy,baseline,labels=[0,1,2],average=None,zero_division=0)[1:])
print('Mean of six individual accuracies:', f'{results.Accuracy.mean():.2%}', '(not ensemble accuracy)')
if RUN_RETRAIN:
    results.to_csv(OUTPUT / 'holdout_results.csv', index=False)
    np.savez_compressed(OUTPUT / 'holdout_probabilities.npz', target=yy, sample_id=test.sample_id.to_numpy(dtype=str), **predictions)
    print('Separate outputs saved. Website selection unchanged.')

              Model  Accuracy  Macro_F1  Sometimes_recall  High_recall    QWK  Macro_AUC  High_detected  High_missed
Logistic Regression    0.9000    0.9014            0.8644       0.9075 0.9169     0.9837            353           36
      Random Forest    0.8907    0.8934            0.8492       0.9126 0.9094     0.9776            355           34
            XGBoost    0.8693    0.8715            0.8475       0.8535 0.8880     0.9699            332           57
             1D CNN    0.9053    0.9071            0.8695       0.9177 0.9214     0.9844            357           32
               LSTM    0.9140    0.9159            0.8864       0.9306 0.9284     0.9857            362           27
           CNN-LSTM    0.8460    0.8486            0.7627       0.8895 0.8748     0.9607            346           43

Training-majority baseline class: 1 | holdout accuracy: 39.33%
Baseline sometimes/high recall: [1. 0.]
Mean of six individual accuracies: 88.76% (not ensemble accuracy)


## 8. Confusion matrices and readable class reports
Matrix rows are actual labels; columns are predicted labels, in order rarely, sometimes, high. A high-stress miss is a class-2 row assigned class 0 or 1. Accuracy alone cannot describe these errors.

In [15]:
for name in NAMES:
    pred = predictions[name].argmax(1)
    print('\n' + name)
    print('Confusion matrix (actual rows, predicted columns):')
    print(confusion_matrix(yy, pred, labels=[0,1,2]))
    print(classification_report(yy, pred, labels=[0,1,2], target_names=['Rarely / never','Sometimes','Usually / always'], digits=3, zero_division=0))


Logistic Regression
Confusion matrix (actual rows, predicted columns):
[[487  34   0]
 [ 41 510  39]
 [  0  36 353]]
                  precision    recall  f1-score   support

  Rarely / never      0.922     0.935     0.929       521
       Sometimes      0.879     0.864     0.872       590
Usually / always      0.901     0.907     0.904       389

        accuracy                          0.900      1500
       macro avg      0.901     0.902     0.901      1500
    weighted avg      0.900     0.900     0.900      1500


Random Forest
Confusion matrix (actual rows, predicted columns):
[[480  41   0]
 [ 52 501  37]
 [  0  34 355]]
                  precision    recall  f1-score   support

  Rarely / never      0.902     0.921     0.912       521
       Sometimes      0.870     0.849     0.859       590
Usually / always      0.906     0.913     0.909       389

        accuracy                          0.891      1500
       macro avg      0.893     0.894     0.893      1500
    weighte

## 9. How the website uses these models
The demo loader reads `models/training_preview.json`, checks CSV/checkpoint hashes, loads the four base families and two qualified replacements, validates exactly 19 inputs, applies each saved encoder, and returns probabilities. The UI shows per-model results. Consensus equally averages the six family probability vectors and then applies argmax; this differs from majority voting. The real-BRFSS server and its operating threshold remain separate.

In [16]:
combined = np.mean(list(predictions.values()), axis=0)
print('Equal-probability ensemble accuracy:', f'{accuracy_score(yy,combined.argmax(1)):.2%}')
print('Ensemble recalls [rarely,sometimes,high]:', recall_score(yy,combined.argmax(1),labels=[0,1,2],average=None))
example = {k: float(test.iloc[0][k]) if k == 'bmi' else int(test.iloc[0][k]) for k in FEATURES}
response = engine.predict(example)
assert len(response['models']) == 6
print('Single-profile API example:')
for name, item in response['models'].items(): print(name, '->', item['label'], '| probabilities:', np.round(item['probabilities'],4))
print('Consensus:', response['consensus']['label'])
assert digest(DATA) == engine.manifest['sha256']
print('Dataset unchanged. No active checkpoint or website selector written by notebook.')

Equal-probability ensemble accuracy: 90.53%
Ensemble recalls [rarely,sometimes,high]: [0.93474088 0.86779661 0.92287918]
Single-profile API example:
Logistic Regression -> Sometimes | probabilities: [0.0025 0.5088 0.4887]
Random Forest -> Usually / always | probabilities: [0.     0.4446 0.5554]
XGBoost -> Sometimes | probabilities: [0.0148 0.5703 0.4149]
1D CNN -> Usually / always | probabilities: [0.     0.4744 0.5256]
LSTM -> Sometimes | probabilities: [2.000e-04 6.167e-01 3.831e-01]
CNN-LSTM -> Usually / always | probabilities: [0.0024 0.3454 0.6522]
Consensus: Usually / always
Dataset unchanged. No active checkpoint or website selector written by notebook.


## 10. Important files and viva answers
| File / folder | Explain it as |
|---|---|
| `model_training/base_models/model_training_data.csv` | Current 10,000-profile educational training CSV |
| `model_training/base_models/training_roles.csv` | Exact 6,300 fitting / 700 development IDs and order |
| `models/current_models/schema.json` | Input names, allowed ranges and target labels |
| `src/training_variants.py` | Exact feature encoder and improved CNN architecture |
| `src/models.py` | Base neural architectures used by LSTM and CNN-LSTM |
| `model_training/base_models/models/` | Four active base families plus preserved earlier checkpoints |
| `model_training/improved_models/models/` | Qualified RF/CNN replacements |
| `model_training/improved_models/overrides.json` | Replacement architecture/settings, paths and hashes |
| `src/training_preview_inference.py` | Loads demo models and produces six predictions |
| `models/training_preview.json` | Chooses the local demo version |
| `models/current_models/` | Separate real-BRFSS model artifacts |
| `archive/previous_experiments/` | Preserved older experiments, outside the current study path |
| `models/path_aliases.json` | Old-to-new paths for historical frozen manifests |

**Why preprocess?** Numeric scales differ; categorical codes are not distances for linear/neural models. Fit transforms only on training rows.

**What is training?** The algorithm adjusts coefficients, tree splits or neural weights to reduce a loss on fitting examples.

**Why separate development and test?** Development can choose epochs. Reusing test results for choosing models biases the reported score; our existing holdout is explicitly reused.

**How did accuracy increase?** First, a different, lower-noise generated target made the demonstration easier. Later, on that same fixed CSV, RF feature engineering/tree tuning and a position-preserving CNN improved their results. Those are distinct changes, not a demonstrated jump in real-person prediction.

**Why recall as well as accuracy?** A model can gain accuracy by ignoring a minority class. We count sometimes/high cases detected and missed as well.

**What does 90% mean here?** About 90% correct on these generated evaluation labels. It does not imply 90% correctness for actual people or a medical diagnosis.

**Can you retrain safely?** Set `RUN_RETRAIN=True`, run all cells, inspect the new `study_run_...` folder, and keep the website selector unchanged unless a separate promotion review qualifies the result.

**Opening this file:** use VS Code with a Python/Jupyter kernel containing numpy, pandas, scikit-learn, xgboost, torch and joblib. The outputs already included can be studied without running anything. This file was executed using the existing `/opt/stress-venv/bin/python` environment; no training package installation was needed.